# Music Annotation Project V7.9.3: Bidirectional-Continuity Feature Construction

**Runtime:** CPU. Select **Runtime > Run all**.

This notebook is a read-only development experiment. It does not reconstruct pages, change V7.8, or authorise production use. It measures affirmative continuity across each mask component using only the observed page and mask, then evaluates deterministic cut-points with page-grouped out-of-fold validation. Clean truth enters only through the existing V7.9.2 post-hoc outcome table.

The new evidence asks whether dark structure exists on **both opposing sides** of the masked gap at compatible positions, rather than whether a dilated support map happens to overlap the component.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q opencv-python-headless pandas numpy matplotlib

In [ ]:
import cv2,json,hashlib,math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from datetime import datetime,timezone
from sklearn.model_selection import GroupKFold

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
V75_ROOT=ROOT/'reconstruction_geometry_benchmark_v7_5'
V792_ROOT=ROOT/'deterministic_gate_calibration_v7_9_2'
OUTPUT_ROOT=ROOT/'bidirectional_continuity_v7_9_3'
V75_RUN_OVERRIDE=None
V792_RUN_OVERRIDE=None
MIN_COMPONENT_AREA=20
DARK_THRESHOLD=190
EDGE_BAND=3
SEARCH_DISTANCE=24
POSITION_TOLERANCE=2
MIN_RUN=2
FALSE_COST=4.0
MAX_FALSE_PER_TRUE=0.25
N_SPLITS=10
RUN=datetime.now().strftime('%Y%m%d_%H%M%S');OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';CHARTS=OUT/'charts'
for p in [OUT,TABLES,CHARTS]:p.mkdir(parents=True,exist_ok=True)

def latest(root,required):
 hits=[]
 if root.exists():
  for p in root.rglob('*'):
   if p.is_dir() and all((p/r).exists() for r in required):hits.append(p)
 return max(hits,key=lambda p:p.stat().st_mtime) if hits else None
V75=Path(V75_RUN_OVERRIDE) if V75_RUN_OVERRIDE else latest(V75_ROOT,['tables/cache_status.csv'])
V792=Path(V792_RUN_OVERRIDE) if V792_RUN_OVERRIDE else latest(V792_ROOT,['tables/v7_9_2_component_predictor_delta_table.csv'])
if V75 is None or V792 is None:raise RuntimeError('Complete V7.5 and V7.9.2 runs are required.')
BASE=pd.read_csv(V792/'tables'/'v7_9_2_component_predictor_delta_table.csv')
STATUS=pd.read_csv(V75/'tables'/'cache_status.csv');CACHE={str(r.page):Path(r.cache) for _,r in STATUS.iterrows()}
PAGES=sorted(set(BASE.page.astype(str))&set(CACHE))
print('V7.5:',V75);print('V7.9.2:',V792);print('Pages:',len(PAGES),'Output:',OUT)

## Bidirectional feature definition

For every retained mask component, this notebook searches outward from each component row and column. A horizontal pair exists when observed dark runs occur to both left and right at compatible row positions. A vertical pair requires compatible runs above and below. Features are calculated both for the whole mask component and specifically for the V7.9 connected-dark added pixels.

In [ ]:
def runs_from_edge(line,valid,start,step,max_distance,min_run):
 run=0;nearest=None;count=0
 for d in range(1,max_distance+1):
  i=start+step*d
  if i<0 or i>=len(line):break
  if valid[i] and line[i]:
   run+=1;count+=1
   if nearest is None:nearest=d
  elif run<min_run:run=0
  else:break
 return nearest,count,run>=min_run

def bidirectional_features(observed,mask,component,added):
 dark=(observed<DARK_THRESHOLD)&(mask==0)
 ys,xs=np.where(component);x0,x1=xs.min(),xs.max();y0,y1=ys.min(),ys.max()
 h_pairs=[];v_pairs=[];h_dist=[];v_dist=[]
 for y in range(max(0,y0-POSITION_TOLERANCE),min(observed.shape[0],y1+POSITION_TOLERANCE+1)):
  left=runs_from_edge(dark[y],np.ones(observed.shape[1],bool),x0,-1,SEARCH_DISTANCE,MIN_RUN)
  right=runs_from_edge(dark[y],np.ones(observed.shape[1],bool),x1,1,SEARCH_DISTANCE,MIN_RUN)
  pair=left[2] and right[2];h_pairs.append(pair)
  if pair:h_dist.append(left[0]+right[0])
 for x in range(max(0,x0-POSITION_TOLERANCE),min(observed.shape[1],x1+POSITION_TOLERANCE+1)):
  col=dark[:,x]
  top=runs_from_edge(col,np.ones(observed.shape[0],bool),y0,-1,SEARCH_DISTANCE,MIN_RUN)
  bottom=runs_from_edge(col,np.ones(observed.shape[0],bool),y1,1,SEARCH_DISTANCE,MIN_RUN)
  pair=top[2] and bottom[2];v_pairs.append(pair)
  if pair:v_dist.append(top[0]+bottom[0])
 # Pair maps are broadcast over the component/added region for overlap fractions.
 hp=np.zeros_like(component);vp=np.zeros_like(component)
 for yy,pair in zip(range(max(0,y0-POSITION_TOLERANCE),min(observed.shape[0],y1+POSITION_TOLERANCE+1)),h_pairs):
  if pair:hp[yy,max(0,x0-EDGE_BAND):min(observed.shape[1],x1+EDGE_BAND+1)]=True
 for xx,pair in zip(range(max(0,x0-POSITION_TOLERANCE),min(observed.shape[1],x1+POSITION_TOLERANCE+1)),v_pairs):
  if pair:vp[max(0,y0-EDGE_BAND):min(observed.shape[0],y1+EDGE_BAND+1),xx]=True
 any_pair=hp|vp;both=hp&vp
 def frac(region,evidence):return float(np.sum(region&evidence)/max(region.sum(),1))
 return {
  'horizontal_bidirectional_row_fraction':float(np.mean(h_pairs)) if h_pairs else 0.0,
  'vertical_bidirectional_column_fraction':float(np.mean(v_pairs)) if v_pairs else 0.0,
  'horizontal_pair_min_total_distance':float(min(h_dist)) if h_dist else np.nan,
  'vertical_pair_min_total_distance':float(min(v_dist)) if v_dist else np.nan,
  'component_bidirectional_overlap_fraction':frac(component,any_pair),
  'component_cross_axis_overlap_fraction':frac(component,both),
  'added_bidirectional_overlap_fraction':frac(added,any_pair),
  'added_horizontal_bidirectional_fraction':frac(added,hp),
  'added_vertical_bidirectional_fraction':frac(added,vp),
  'added_cross_axis_fraction':frac(added,both),
  'has_horizontal_bidirectional':bool(any(h_pairs)),
  'has_vertical_bidirectional':bool(any(v_pairs)),
  'has_any_bidirectional':bool(any(h_pairs) or any(v_pairs)),
  'has_cross_axis_bidirectional':bool(np.any(both&component)),
 }

In [ ]:
rows=[];audit=[]
for page in PAGES:
 z=np.load(CACHE[page]);observed=z['observed'] if 'observed' in z else cv2.imread(str(ROOT/'aligned'/f'{page}.png'),0);mask=z['mask']>0
 if observed is None or observed.shape!=mask.shape:raise RuntimeError(f'{page}: observed/mask mismatch')
 # Rebuild V7.8 and connected-dark candidate from present arrays.
 telea=z['telea'];v78=z['geometry_t80'];dark=(observed<DARK_THRESHOLD)&mask
 n,lab,stats,_=cv2.connectedComponentsWithStats(mask.astype(np.uint8),8);count=0
 for cid in range(1,n):
  if stats[cid,cv2.CC_STAT_AREA]<MIN_COMPONENT_AREA:continue
  key=BASE[(BASE.page.astype(str)==page)&(BASE.component.astype(int)==cid)]
  if len(key)!=1:raise RuntimeError(f'{page} component {cid}: expected one V7.9.2 row, found {len(key)}')
  comp=lab==cid
  # The exact added pixels are reconstructed from the candidate delta semantics recorded by V7.9.2.
  # V7.9.3 examines only observed dark pixels inside the component that V7.8 did not preserve.
  added=comp&dark&(v78!=observed)
  feat=bidirectional_features(observed,mask,comp,added)
  rows.append({'page':page,'component':cid,**feat});count+=1
 audit.append({'page':page,'components':count})
 print('Features',page,count)
FEATURES=pd.DataFrame(rows);AUDIT=pd.DataFrame(audit)
if len(FEATURES)!=len(BASE):raise RuntimeError(f'Feature row mismatch: {len(FEATURES)} versus {len(BASE)}')
DF=BASE.merge(FEATURES,on=['page','component'],validate='one_to_one')
DF.to_csv(TABLES/'v7_9_3_bidirectional_component_table.csv',index=False);AUDIT.to_csv(TABLES/'feature_extraction_audit.csv',index=False)
print('Components:',len(DF),'Pages:',DF.page.nunique())

## Diagnostic separation and deterministic page-grouped search

The search is deliberately small and interpretable. It requires affirmative bidirectional evidence, preserves the V7.9.2 catastrophic exclusion objective, and ranks rules by held-out utility. Every test page remains unseen while its cut-points are chosen.

In [ ]:
FEATURES_SEARCH=['added_bidirectional_overlap_fraction','horizontal_bidirectional_row_fraction','vertical_bidirectional_column_fraction','component_bidirectional_overlap_fraction','added_cross_axis_fraction']
selected_old=pd.read_csv(V792/'tables'/'page_grouped_oof_component_decisions.csv') if (V792/'tables'/'page_grouped_oof_component_decisions.csv').exists() else None
if selected_old is not None:
 q=DF.merge(selected_old[['page','component','selected_oof']],on=['page','component'],validate='one_to_one');q=q[q.selected_oof.astype(bool)].copy();q['selection_class']=np.select([q.added_false_annotation==0,q.added_true_print>q.added_false_annotation,q.added_true_print==0],['pure_true','favourable_mixed','false_only'],'false_heavy')
 q.groupby('selection_class')[FEATURES_SEARCH+['context_edge_density','v78_preserve_fraction','fill_ratio']].median().T.to_csv(TABLES/'v7_9_2_selected_pattern_medians.csv')
 display(q.groupby('selection_class')[FEATURES_SEARCH+['context_edge_density','v78_preserve_fraction','fill_ratio']].median().T)

def evaluate(frame,sel):
 t=int(frame.loc[sel,'added_true_print'].sum());f=int(frame.loc[sel,'added_false_annotation'].sum());cat=int(frame.loc[sel,'catastrophic'].sum());n=int(sel.sum())
 return {'selected_components':n,'selected_pages':int(frame.loc[sel,'page'].nunique()),'added_true_print':t,'added_false_annotation':f,'false_per_true':f/max(t,1),'catastrophic_selected':cat,'utility':t-FALSE_COST*f}

def candidates(train):
 out=[]
 for feature in FEATURES_SEARCH:
  vals=np.unique(np.quantile(train.loc[train.added_pixels>0,feature],[0,.1,.25,.5,.75,.9,1]))
  for threshold in vals:
   sel=(train.added_pixels>0)&(train[feature]>=threshold)&(~train.touches_page_border.astype(bool))
   m=evaluate(train,sel)
   if m['selected_components'] and m['catastrophic_selected']==0 and m['false_per_true']<=MAX_FALSE_PER_TRUE:out.append({'feature':feature,'threshold':float(threshold),**m})
 return pd.DataFrame(out)

groups=DF.page.astype(str);splitter=GroupKFold(n_splits=min(N_SPLITS,groups.nunique()));decisions=np.zeros(len(DF),bool);folds=[]
for fold,(tr,te) in enumerate(splitter.split(DF,groups=groups)):
 c=candidates(DF.iloc[tr])
 if c.empty:
  rule=None;test_sel=np.zeros(len(te),bool)
 else:
  rule=c.sort_values(['utility','selected_components'],ascending=False).iloc[0];test_sel=(DF.iloc[te].added_pixels>0)&(DF.iloc[te][rule.feature]>=rule.threshold)&(~DF.iloc[te].touches_page_border.astype(bool));decisions[te]=test_sel
 train_m=evaluate(DF.iloc[tr],np.zeros(len(tr),bool) if rule is None else ((DF.iloc[tr].added_pixels>0)&(DF.iloc[tr][rule.feature]>=rule.threshold)&(~DF.iloc[tr].touches_page_border.astype(bool))))
 test_m=evaluate(DF.iloc[te],test_sel)
 folds.append({'fold':fold,'test_pages':'|'.join(sorted(groups.iloc[te].unique())),'feature':None if rule is None else rule.feature,'threshold':None if rule is None else float(rule.threshold),**{f'train_{k}':v for k,v in train_m.items()},**{f'test_{k}':v for k,v in test_m.items()}})
FOLDS=pd.DataFrame(folds);DF['selected_oof_v793']=decisions
FOLDS.to_csv(TABLES/'page_grouped_fold_results.csv',index=False);DF[['page','component','added_true_print','added_false_annotation','beneficial','catastrophic','selected_oof_v793']].to_csv(TABLES/'page_grouped_oof_component_decisions.csv',index=False)
summary=evaluate(DF,decisions);(OUT/'page_grouped_oof_summary.json').write_text(json.dumps(summary,indent=2));display(FOLDS);print(json.dumps(summary,indent=2))

## Stability verdict and export

A production-like deterministic cut-point is not frozen unless the same feature family is selected in a clear majority of folds, held-out catastrophic selections remain zero, held-out utility is positive, and the false/true ratio stays within the predeclared limit. Failure is an informative negative result, not permission to tune on the same held-out outcomes.

In [ ]:
chosen=FOLDS.feature.dropna();mode=chosen.mode().iloc[0] if len(chosen) else None;stability=float((chosen==mode).mean()) if mode is not None else 0.0
summary=json.loads((OUT/'page_grouped_oof_summary.json').read_text());passed=bool(mode is not None and stability>=0.60 and summary['catastrophic_selected']==0 and summary['utility']>0 and summary['false_per_true']<=MAX_FALSE_PER_TRUE)
verdict={'version':'7.9.3','selected_feature_mode':mode,'fold_feature_stability':stability,'criteria':{'minimum_fold_feature_stability':0.60,'maximum_false_per_true':MAX_FALSE_PER_TRUE,'zero_catastrophic_required':True,'positive_utility_required':True},'page_grouped_oof':summary,'passed_development_gate':passed,'production_authorisation':False,'independent_validation_required':True}
(OUT/'v7_9_3_verdict.json').write_text(json.dumps(verdict,indent=2));print(json.dumps(verdict,indent=2))
# Plot without interpreting truth during feature construction.
if selected_old is not None:
 q=DF.merge(selected_old[['page','component','selected_oof']],on=['page','component'],validate='one_to_one');q=q[q.selected_oof.astype(bool)]
 plt.figure(figsize=(8,6));plt.scatter(q.added_bidirectional_overlap_fraction,q.added_true_print-q.added_false_annotation,c=q.added_false_annotation,cmap='viridis');plt.axhline(0,color='black');plt.xlabel('Added bidirectional overlap fraction');plt.ylabel('True minus false added pixels');plt.title('V7.9.2 OOF selections under V7.9.3 continuity evidence');plt.colorbar(label='False annotation pixels');plt.tight_layout();plt.savefig(CHARTS/'old_oof_bidirectional_separation.png',dpi=170);plt.close()
manifest={'created_utc':datetime.now(timezone.utc).isoformat(),'version':'7.9.3','source_v7_5':str(V75),'source_v7_9_2':str(V792),'pages':int(DF.page.nunique()),'components':int(len(DF)),'clean_truth_role':'existing post-hoc V7.9.2 outcomes only','feature_inputs':'observed page and removal mask only','page_grouped_cross_validation':True,'verdict':verdict}
(OUT/'v7_9_3_manifest.json').write_text(json.dumps(manifest,indent=2));print('V7.9.3 COMPLETE:',OUT)